# Step 2: Generate Embeddings with all-MiniLM-L6-v2

## Goal
Convert each paper's abstract into a 384-dimensional dense vector
using the `all-MiniLM-L6-v2` sentence embedding model.

## Input
- `/kaggle/input/fairsearch-qbio-processed-Raj-Jici-YB/qbio_papers.json`
- 55,301 records from the current Kaggle arXiv snapshot

## Output
- `/kaggle/working/qbio_embeddings.npy` — embedding matrix, shape `(num_records, 384)`
- `/kaggle/working/embedding_info.json` — model metadata + ordered paper_ids

## Key Rule
`embeddings[i]` must always correspond to `qbio_papers[i]`.
Do NOT shuffle or re-sort either file.

In [1]:
# Install required packages
!pip install sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.3 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

## Cell 2: Import Libraries and Load Model

Load the `all-MiniLM-L6-v2` model via `sentence-transformers`.
The model will automatically use GPU if available.

In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── Input / Output paths ──────────────────────────────────────────────────────
INPUT_PATH = "/kaggle/input/datasets/yanbochen928/fairsearch-qbio-processed-raj-jici-yb/qbio_papers.json"
OUTPUT_EMB_PATH  = "/kaggle/working/qbio_embeddings.npy"
OUTPUT_INFO_PATH = "/kaggle/working/embedding_info.json"

# ── Load model ────────────────────────────────────────────────────────────────
# all-MiniLM-L6-v2: distilled BERT optimized for semantic similarity
# automatically uses GPU if available
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Device: {model.device}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: all-MiniLM-L6-v2
Device: cuda:0


## Cell 3: Load Papers and Extract Abstracts

Load `qbio_papers.json` and extract abstracts in order.
The order must be preserved so that `embeddings[i]` always corresponds to `qbio_papers[i]`.

In [3]:
# ── Load qbio_papers.json ─────────────────────────────────────────────────────
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    qbio_papers = json.load(f)

print(f"Total papers loaded: {len(qbio_papers):,}")

# ── Extract abstracts and paper_ids in order ──────────────────────────────────
# Order must be preserved: abstracts[i] corresponds to qbio_papers[i]
abstracts = [paper["abstract"] for paper in qbio_papers]
paper_ids = [paper["paper_id"] for paper in qbio_papers]

print(f"Total abstracts extracted: {len(abstracts):,}")
print(f"\nSpot-check first 3 paper_ids:")
for i in range(3):
    print(f"  [{i}] {paper_ids[i]} | abstract[:60]: {abstracts[i][:60]}...")

Total papers loaded: 55,301
Total abstracts extracted: 55,301

Spot-check first 3 paper_ids:
  [0] 0704.0021 | abstract[:60]: Spatiotemporal pattern formation in a product-activated enzy...
  [1] 0704.0034 | abstract[:60]: This is a supplement to the paper arXiv:q-bio/0701050, conta...
  [2] 0704.0036 | abstract[:60]: The multisite phosphorylation-dephosphorylation cycle is a m...


## Cell 4: Generate Embeddings

Encode all 55,301 abstracts using `all-MiniLM-L6-v2` in batches of 64.
GPU parallelism significantly speeds up this step.
Output shape: `(num_records, 384)`

In [4]:
# ── Generate embeddings in batches ────────────────────────────────────────────
# batch_size=64 takes full advantage of GPU parallelism
# show_progress_bar=True displays a tqdm progress bar
embeddings = model.encode(
    abstracts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbedding complete.")
print(f"Shape: {embeddings.shape}")
print(f"Expected: ({len(abstracts)}, 384)")

Batches:   0%|          | 0/865 [00:00<?, ?it/s]


Embedding complete.
Shape: (55301, 384)
Expected: (55301, 384)


## Cell 5: Save Output Files

Save the embedding matrix as `qbio_embeddings.npy` and
save the metadata (model name, dimension, record count, paper_id order)
as `embedding_info.json`.

In [5]:
# ── Save embedding matrix ─────────────────────────────────────────────────────
np.save(OUTPUT_EMB_PATH, embeddings)
print(f"Saved: {OUTPUT_EMB_PATH}")

# ── Save embedding_info.json ──────────────────────────────────────────────────
# This file tells Step 3 (Raj) which paper_id corresponds to which row
embedding_info = {
    "model": "all-MiniLM-L6-v2",
    "embedding_dim": 384,
    "num_records": len(paper_ids),
    "order": "same as qbio_papers.json",
    "paper_ids": paper_ids
}

with open(OUTPUT_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(embedding_info, f, ensure_ascii=False, indent=2)
print(f"Saved: {OUTPUT_INFO_PATH}")

Saved: /kaggle/working/qbio_embeddings.npy
Saved: /kaggle/working/embedding_info.json


## Cell 6: Validation

Verify the output files are correct:
- Embedding matrix shape should be `(num_records, 384)`
- First 3 paper_ids from `embedding_info.json`
- Sample vector (first 5 dimensions of `embeddings[0]`)
- Output file sizes on disk

In [6]:
import os

# ── Reload and verify embedding matrix ───────────────────────────────────────
embeddings_loaded = np.load(OUTPUT_EMB_PATH)
print(f"=== Validation ===")
print(f"Embedding shape      : {embeddings_loaded.shape}")
print(f"Expected             : ({len(paper_ids)}, 384)")

# ── Verify embedding_info.json ────────────────────────────────────────────────
with open(OUTPUT_INFO_PATH, "r", encoding="utf-8") as f:
    info = json.load(f)

print(f"\nModel                : {info['model']}")
print(f"Embedding dim        : {info['embedding_dim']}")
print(f"Num records          : {info['num_records']}")
print(f"First 3 paper_ids    : {info['paper_ids'][:3]}")

# ── Sample vector ─────────────────────────────────────────────────────────────
print(f"\nSample vector (embeddings[0], first 5 dims):")
print(f"  {embeddings_loaded[0][:5]}")

# ── File sizes ────────────────────────────────────────────────────────────────
emb_size  = os.path.getsize(OUTPUT_EMB_PATH)  / (1024 * 1024)
info_size = os.path.getsize(OUTPUT_INFO_PATH) / (1024 * 1024)
print(f"\nFile sizes:")
print(f"  qbio_embeddings.npy  : {emb_size:.2f} MB")
print(f"  embedding_info.json  : {info_size:.2f} MB")

=== Validation ===
Embedding shape      : (55301, 384)
Expected             : (55301, 384)

Model                : all-MiniLM-L6-v2
Embedding dim        : 384
Num records          : 55301
First 3 paper_ids    : ['0704.0021', '0704.0034', '0704.0036']

Sample vector (embeddings[0], first 5 dims):
  [-0.0347089  -0.14605787 -0.04744145 -0.03489042 -0.03596909]

File sizes:
  qbio_embeddings.npy  : 81.01 MB
  embedding_info.json  : 0.96 MB
